In [1]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset

In [2]:
model = torch.load("../models/gnn_model.pt")

In [3]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [4]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads all graphs to a consistent node feature dimension."""

    def __init__(self, dataset, target_dim: int | None = None):
        self.dataset = dataset
        # Compute max dimension across entire dataset if not provided
        if target_dim is None:
            self.target_dim = self._compute_max_dim()
        else:
            self.target_dim = target_dim

    def _compute_max_dim(self) -> int:
        """Find the max node feature dimension across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            if hasattr(data, "x") and data.x.dim() > 1:
                max_dim = max(max_dim, data.x.shape[1])
        return max_dim

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        if hasattr(data, "x") and data.x.shape[1] < self.target_dim:
            pad_size = self.target_dim - data.x.shape[1]
            # Create a new copy to avoid mutating cache
            data = data.clone()
            data.x = torch.nn.functional.pad(data.x, (0, pad_size), mode="constant", value=0)
        return data

    def __len__(self) -> int:
        return len(self.dataset)

In [5]:
def build_train_val_test_loaders_two_stage(
    pt_paths: list[str],
    train_split: float = 0.8,
    val_within_train: float = 0.1,
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, DataLoader, DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    generator = torch.Generator().manual_seed(seed)
    primary_train_len = max(1, int(len(padded_dataset) * train_split))
    test_len = max(1, len(padded_dataset) - primary_train_len)
    while primary_train_len + test_len > len(padded_dataset):
        primary_train_len -= 1

    primary_train, test_ds = random_split(
        padded_dataset,
        [primary_train_len, test_len],
        generator=generator,
    )

    val_len = max(1, int(len(primary_train) * val_within_train))
    real_train_len = max(1, len(primary_train) - val_len)
    train_ds, val_ds = random_split(
        primary_train,
        [real_train_len, val_len],
        generator=generator,
    )

    pin_mem = torch.cuda.is_available()

    return (
        dataset,
        padded_dataset,
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [6]:
family = "quansistor"

In [7]:
data_paths = collect_pt_paths("../outputs/data", family=family)

In [8]:
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 5000 .pt files for dataset.


In [9]:
for path in data_paths[:5]:  # Print first 5 paths as a sanity check
    print(path)

..\outputs\data\encoding_data_pennylane\quansistor\quansistor_Q10_L11_S1062816626.pt
..\outputs\data\encoding_data_pennylane\quansistor\quansistor_Q10_L11_S1222712210.pt
..\outputs\data\encoding_data_pennylane\quansistor\quansistor_Q10_L11_S1434058421.pt
..\outputs\data\encoding_data_pennylane\quansistor\quansistor_Q10_L11_S1491460037.pt
..\outputs\data\encoding_data_pennylane\quansistor\quansistor_Q10_L11_S1861211939.pt


In [10]:
dataset, padded_data, train_loader, val_loader, test_loader, node_in_dim, global_in_dim = build_train_val_test_loaders_two_stage(
    data_paths,  # Use first 10 paths for demonstration
    train_split=0.8,
    val_within_train=0.1,
    batch_size=32,
    seed=42,
    global_feature_variant="binned",
    node_feature_backend_variant=None,
)

In [11]:
padded_data

In [12]:
dataset

QuantumCircuitGraphDataset(5000)

In [13]:
for i, sample in enumerate(dataset):
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=70, edges=110, x_shape=(70, 23), y=7.637660503387451
   global features shape: torch.Size([1, 302])
1: nodes=70, edges=110, x_shape=(70, 23), y=7.66350793838501
   global features shape: torch.Size([1, 302])
2: nodes=70, edges=110, x_shape=(70, 23), y=7.81531286239624
   global features shape: torch.Size([1, 302])
3: nodes=70, edges=110, x_shape=(70, 23), y=7.698472023010254
   global features shape: torch.Size([1, 302])
4: nodes=70, edges=110, x_shape=(70, 23), y=7.5361762046813965
   global features shape: torch.Size([1, 302])
5: nodes=70, edges=110, x_shape=(70, 23), y=7.518194675445557
   global features shape: torch.Size([1, 302])
6: nodes=70, edges=110, x_shape=(70, 23), y=7.702983856201172
   global features shape: torch.Size([1, 302])
7: nodes=70, edges=110, x_shape=(70, 23), y=7.461328029632568
   global features shape: torch.Size([1, 302])
8: nodes=70, edges=110, x_shape=(70, 23), y=7.564272880554199
   global features shape: torch.Size([1, 302])
9: nodes=70, edges=1

In [14]:
MASTER_GATE_TYPES = [
    "input", "measurement",
    "h", "s", "t", "id",
    "rx", "ry", "rz",
    "cx",
    "qx", "qy", "haar",
]

FAMILY_GATE_TYPES = {
    "random": ["input", "measurement", "rx", "ry", "rz", "cx"],
    "clifford": ["input", "measurement", "h", "s", "t", "id", "cx"],
    "haar": ["input", "measurement", "haar"],
    "quansistor": ["input", "measurement", "qx", "qy"],
}

In [15]:
class FamilyNodeProjector:
    def __init__(self, family: str):
        self.family = family
        self.master_gate_types = MASTER_GATE_TYPES
        self.family_gate_types = FAMILY_GATE_TYPES[family]
        self.keep_gate_idx = [
            self.master_gate_types.index(name) for name in self.family_gate_types
        ]
        self.n_gate_master = len(self.master_gate_types)

    def __call__(self, data: Data) -> Data:
        gate_part = data.x[:, :self.n_gate_master]
        qubit_part = data.x[:, self.n_gate_master:]

        out = data.clone()
        out.x = torch.cat([gate_part[:, self.keep_gate_idx], qubit_part], dim=1)
        return out

In [16]:
projector = FamilyNodeProjector(family)
for i, sample in enumerate(dataset):
    sample = projector(sample)
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=70, edges=110, x_shape=(70, 14), y=7.637660503387451
   global features shape: torch.Size([1, 302])
1: nodes=70, edges=110, x_shape=(70, 14), y=7.66350793838501
   global features shape: torch.Size([1, 302])
2: nodes=70, edges=110, x_shape=(70, 14), y=7.81531286239624
   global features shape: torch.Size([1, 302])
3: nodes=70, edges=110, x_shape=(70, 14), y=7.698472023010254
   global features shape: torch.Size([1, 302])
4: nodes=70, edges=110, x_shape=(70, 14), y=7.5361762046813965
   global features shape: torch.Size([1, 302])
5: nodes=70, edges=110, x_shape=(70, 14), y=7.518194675445557
   global features shape: torch.Size([1, 302])
6: nodes=70, edges=110, x_shape=(70, 14), y=7.702983856201172
   global features shape: torch.Size([1, 302])
7: nodes=70, edges=110, x_shape=(70, 14), y=7.461328029632568
   global features shape: torch.Size([1, 302])
8: nodes=70, edges=110, x_shape=(70, 14), y=7.564272880554199
   global features shape: torch.Size([1, 302])
9: nodes=70, edges=1

In [17]:
sample

Data(
  x=[70, 14],
  edge_index=[2, 110],
  y=[1],
  global_features=[1, 302],
  num_qubits=10,
  gate_counts={
    qx_a_bin_0=0,
    qx_a_bin_1=0,
    qx_a_bin_10=0,
    qx_a_bin_11=0,
    qx_a_bin_12=0,
    qx_a_bin_13=0,
    qx_a_bin_14=1,
    qx_a_bin_15=0,
    qx_a_bin_16=2,
    qx_a_bin_17=0,
    qx_a_bin_18=0,
    qx_a_bin_19=1,
    qx_a_bin_2=2,
    qx_a_bin_20=1,
    qx_a_bin_21=1,
    qx_a_bin_22=1,
    qx_a_bin_23=1,
    qx_a_bin_24=0,
    qx_a_bin_25=0,
    qx_a_bin_26=0,
    qx_a_bin_27=0,
    qx_a_bin_28=1,
    qx_a_bin_29=0,
    qx_a_bin_3=0,
    qx_a_bin_30=2,
    qx_a_bin_31=0,
    qx_a_bin_32=1,
    qx_a_bin_33=0,
    qx_a_bin_34=0,
    qx_a_bin_35=1,
    qx_a_bin_36=0,
    qx_a_bin_37=0,
    qx_a_bin_38=0,
    qx_a_bin_39=0,
    qx_a_bin_4=0,
    qx_a_bin_40=0,
    qx_a_bin_41=0,
    qx_a_bin_42=1,
    qx_a_bin_43=0,
    qx_a_bin_44=1,
    qx_a_bin_45=1,
    qx_a_bin_46=2,
    qx_a_bin_47=2,
    qx_a_bin_48=1,
    qx_a_bin_49=2,
    qx_a_bin_5=0,
    qx_a_bin_6=1,
 

In [18]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "predictions" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [19]:
family = "clifford"

In [20]:
pred_data_paths = collect_pred_paths("../outputs/data", family=family)

In [21]:
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 8750 .pt files for dataset.


In [22]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [23]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

In [24]:
for i, sample in enumerate(dataset):
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
1: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
2: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
3: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
4: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
5: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
6: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
7: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
8: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
9: nodes=227, edges=276, x_shape=(227, 25)
   global features shape: torch.Size([1, 7])
...showing first 10 samples only


In [25]:
sample = dataset[0]
print(sample)

Data(
  x=[227, 25],
  edge_index=[2, 276],
  y=[1],
  global_features=[1, 7],
  num_qubits=12,
  gate_counts={
    CNOT_count=61,
    H_count=32,
    I_count=34,
    S_count=56,
    T_count=20,
  },
  meta={
    cid='clifford_Q12_L11_S1204654993',
    family='clifford',
    n_qubits=12,
    n_layers=11,
    seed=1204654993,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)


tensor([nan])